# Day 34: Foreign Keys Check Demo

This notebook demonstrates how to validate **foreign key relationships** between tables using the data quality framework.

## Overview
Foreign keys ensure referential integrity by verifying that values in a child table's column exist in the parent table's referenced column.

## What This Demo Covers
* Loading test data from CSV files (customers and orders)
* Checking referential integrity between orders and customers
* Identifying orphaned records (orders with invalid customer references)
* Using the `check_referential_integrity()` and `get_orphaned_rows()` functions
* Saving results to a Delta table for tracking and reporting

## Test Scenario
* **Parent Table**: Customers (customer_id: 101, 102)
* **Child Table**: Orders (with customer_id foreign key)
* **Expected Issue**: Some orders reference non-existent customers or have null customer_id

## Key Functions
* `check_referential_integrity(child_df, child_key, parent_df, parent_key)` - Validates foreign key constraint
* `get_orphaned_rows(child_df, child_key, parent_df, parent_key)` - Returns rows that violate the constraint

## Output
* Foreign key validation report showing orphaned records and pass/fail status
* Results saved to `workspace.default.foreign_key_report` table

In [0]:

"""
Foreign Keys Check Demo

This script demonstrates how to validate foreign key relationships between tables.
It checks that all customer_id values in the orders table exist in the customers table.

Foreign Key Relationship:
    orders.customer_id -> customers.customer_id
    
Expected Results:
    - Some orders will have invalid customer_id (999) or null values
    - These orphaned records will be identified and displayed
"""

# ============================================================================
# 1. SETUP: Import required libraries
# ============================================================================
import sys
import yaml
import os

# ============================================================================
# 2. DYNAMIC PATH CONFIGURATION
# ============================================================================
# Get the base repository path dynamically to avoid hard-coded paths
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
# Go up three levels: day34 -> phase2_dq_framework -> notebooks -> data-quality-testing (base)
base_path = os.path.dirname(os.path.dirname(os.path.dirname(notebook_path)))

# ============================================================================
# 3. IMPORT DQ CHECK FUNCTIONS
# ============================================================================
# Add DQ checks module to Python path
checks_path = os.path.join("/Workspace", base_path.lstrip("/"), "src/checks")
sys.path.append(checks_path)
from dq_checks import check_referential_integrity, get_orphaned_rows

# ============================================================================
# 4. LOAD CONFIGURATION
# ============================================================================
# Load YAML configuration file containing foreign key definitions
config_path = os.path.join("/Workspace", base_path.lstrip("/"), "src/config/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

# ============================================================================
# 5. LOAD TEST DATA - PARENT TABLE (CUSTOMERS)
# ============================================================================
# Load customers test data from CSV file
# This is the parent/referenced table in the foreign key relationship
customers_path = os.path.join("/Workspace", base_path.lstrip("/"), "tests/test_data/customers.csv")
customers = spark.read.option("header", True).option("inferSchema", True).csv(customers_path)

print("📊 Customers Data:")
customers.display()

# ============================================================================
# 6. LOAD TEST DATA - CHILD TABLE (ORDERS)
# ============================================================================
# Load orders test data from CSV file
# This is the child/referencing table containing the foreign key column
orders_path = os.path.join("/Workspace", base_path.lstrip("/"), "tests/test_data/orders.csv")
orders = spark.read.option("header", True).option("inferSchema", True).csv(orders_path)

print("📊 Orders Data:")
orders.display()

# ============================================================================
# 7. RUN FOREIGN KEY CHECK
# ============================================================================
# Check referential integrity: Verify that all customer_id values in orders
# exist in the customers table
# Returns: Dictionary with check results including pass/fail status and counts
result = check_referential_integrity(orders, "customer_id", customers, "customer_id")
print(result)

# ============================================================================
# 8. IDENTIFY ORPHANED RECORDS
# ============================================================================
# Get all orders that have customer_id values not found in the customers table
# These are "orphaned" records that violate the foreign key constraint
orphans = get_orphaned_rows(orders, "customer_id", customers, "customer_id")
print("\n🚨 Orphaned Orders (no matching customer):")
orphans.display()

In [0]:
# ============================================================================
# 9. SAVE RESULTS TO DELTA TABLE
# ============================================================================
# Convert the foreign key check result dictionary to a DataFrame
# This allows us to track and analyze foreign key violations over time
result_data = [{
    "check": result["check"],
    "child_key": result["child_key"],
    "total_rows": result["total_rows"],
    "orphan_count": result["orphan_count"],
    "passed": result["passed"]
}]

results_df = spark.createDataFrame(result_data)

# Display the results DataFrame before saving
print("\n📋 Foreign Key Check Results Summary:")
results_df.display()

# Save results to Delta table for tracking and reporting
# Table: workspace.default.foreign_key_report
results_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.foreign_key_report")

print("\n✅ Results saved to 'workspace.default.foreign_key_report' table")

In [0]:
# ============================================================================
# 10. VERIFY SAVED RESULTS
# ============================================================================
# Read back the saved foreign key report from Delta table to verify persistence
df = spark.table("workspace.default.foreign_key_report")

print("📊 Reading back from Delta table: workspace.default.foreign_key_report")
df.display()